# Lab 12 · Agent skills: reusable instructions on demand

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Use a reusable instruction

A skill contains instructions for a particular task. Here we select a small instruction manually and include it in a model prompt. The full exercise adds a catalog, model selection, and validation.

In [ ]:
skill = "Explain a term using one short sentence and a simple example."
task = "Explain what a rainbow is."
print("Loaded instruction:", skill)
print("Answer:", ask(f"Instructions: {skill}\nTask: {task}"))

**Next:** Next, select and load a skill for reviewing a saga.

## Package a saga review as a skill
**Start with a problem:** Repeating every procedure in every prompt makes instructions harder to maintain and can include material irrelevant to the task.

**In this lab:** Choose a relevant skill from short descriptions, load its procedure, and check the proposed saga actions.

**New:** an agent sees skill names and descriptions, selects one, then loads only its body: **progressive disclosure**. A skill is a reusable procedure, not a guarantee that a model follows it.

The domain exercise revisits checkout sagas. When payment fails, compensate completed local writes in reverse order: release reserved stock, then reject the pending order. A completed read needs no undo, and a failed authorization is not a completed charge to refund. If payment succeeds, keep the completed order. Sagas provide no isolation: other services may see intermediate states.

**At a glance**

![Lab 12 at a glance: Skill catalog (names + descriptions), then Pick a skill (name only), then Load instructions (SKILL.md body), then Review saga (per case), then Check the ADR (format + compensation order). Then back to Review saga: one repair, then stop.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/12.svg)

## Think about the example

Checkout creates an order, verifies the customer, reserves stock, and requests payment. A failed authorization is not a completed charge; completed writes may need compensating actions while other services can observe intermediate states.

**Concepts:** Saga compensation in reverse completion order; Read-only steps need no undo; No isolation and intermediate states; Skill discovery and bounded repair.

Use the scenario above to consider these questions before running the code:

1. Why is a failed card authorization not a completed charge to refund?
2. Which intermediate states can other services see before a saga finishes?

## The idea in context

Progressive disclosure shows skill descriptions first and loads instructions only when a task needs them. Host code validates the resulting proposal and bounds repairs.

A saga coordinates local transactions. In this reduced ShopEasy case, failed payment releases reserved stock before rejecting the earlier order; the read-only customer check needs no undo. The code proposes a plan, without executing real transactions.

| Key term | Meaning |
| --- | --- |
| **Skill** | A named reusable procedure selected by its description. |
| **Progressive disclosure** | Load full instructions only after selecting a skill. |
| **Saga compensation** | A business action that counteracts a completed local step. |

## Pattern · Skill Library Evolution

A catalog makes procedures reusable and discoverable. This lab demonstrates selection and use of a saga-review procedure, not automatic skill creation. Keep the scope narrow: a compensation plan is a proposal, not authority to execute a refund or stock change.

## Main exercise at a glance

1. Read an inline SKILL.md example
2. Build a small catalog
3. Select before loading
4. Check shape and compensation order
5. Review failed and successful checkouts

## Step 1 · Read an inline SKILL.md example

The visible text stands in for `SKILL.md`. The small parser handles our controlled header format. The skill explains the domain procedure; a later checker verifies its proposed actions.

A skill has a small header between `---` markers and an instruction body. The helper splits those parts; this example uses a deliberately simple header format.

In [ ]:
def read_skill(text):
    _, header, body = text.split("---", 2)
    fields = {}
    for line in header.strip().splitlines():
        name, value = line.split(": ", 1)
        fields[name] = value
    return fields, body.strip()

adr_skill = """---
name: saga-adr
description: Review a checkout saga and propose ordered compensations in a three-line ADR.
---
If payment fails, choose compensate; reverse completed writes, skipping read-only steps.
create order is undone by reject order; reserve stock by release stock; verify customer only reads.
If payment succeeds, choose complete with no compensations. Never refund a failed authorization.
Return Context, Decision, Consequence. Consequence contains only semicolon-separated actions, or none.
In Context, mention intermediate states: sagas do not provide isolation. Propose actions; do not execute."""
fields, body = read_skill(adr_skill)
print(fields)
print(body)

## Step 2 · Build a small catalog

The other procedures review bounded contexts and read patterns. The model sees only names and descriptions during selection. It has not yet received any compensation instructions.

In [ ]:
read_skill_text = """---
name: read-pattern-review
description: Choose API composition for ID lookups or CQRS for searchable cross-service views.
---
Use owner APIs or published events, never read another service's private tables."""
boundary_skill = """---
name: boundary-review
description: Review ShopEasy boundaries using business language and data ownership.
---
Prefer business capabilities, cohesive models, private data, and independent changes."""
skills = {}
for text in [adr_skill, read_skill_text, boundary_skill]:
    fields, body = read_skill(text)
    skills[fields["name"]] = {"description": fields["description"], "body": body}
catalog_lines = []
for name, skill in skills.items():
    catalog_lines.append(f"- {name}: {skill['description']}")
catalog = "\n".join(catalog_lines)
print(catalog)

## Step 3 · Select before loading

The task asks about a failed checkout, so `saga-adr` is the relevant procedure. Our code accepts only a known skill name before revealing its instructions. The model selects a skill name.

In [ ]:
task = "ShopEasy created an order, verified the customer, reserved stock, then payment failed. Review compensation."
prompt = f"Available skills:\n{catalog}\nTask: {task}\nReply with the one skill name only."
choice = ask(prompt).strip()
if choice not in skills:
    raise ValueError(f"Unknown skill {choice!r}")
print("Chosen:", choice)
print("Loaded instructions:\n" + skills[choice]["body"])

## Step 4 · Check shape and compensation order

The checker requires three labelled lines, the expected decision, and exactly the expected ordered actions. It rejects `reject order; release stock`, because that is not reverse completion order. A successful checkout must contain `none`, not an unnecessary compensation. A human still checks the completeness of the supplied saga model.

The checker verifies the labels and the exact compensation order. It does not execute the compensation or establish that the business policy is correct.

In [ ]:
def adr_problems(adr, expected, actions=()):
    lines = adr.strip().splitlines()
    fields = {}
    for line in lines:
        if ":" in line:
            key, value = line.split(":", 1)
            fields[key.strip()] = value.strip()
    problems = []
    if len(lines) != 3 or list(fields) != ["Context", "Decision", "Consequence"]:
        problems.append("Use exactly Context, Decision, Consequence, one nonempty line each")
    for key in ("Context", "Decision", "Consequence"):
        if not fields.get(key):
            problems.append("Every field needs text")
            break
    if fields.get("Decision", "").strip().lower() != expected:
        problems.append(f"Decision must contain only {expected!r}")
    given = []
    for action in fields.get("Consequence", "").split(";"):
        given.append(action.strip().lower())
    if given != (list(actions) if actions else ["none"]):
        problems.append(f"Consequence must list ordered actions {list(actions) or ['none']}")
    return problems

print(adr_problems("Context: Pending.\nDecision: compensate\nConsequence: reject order; release stock",
                   "compensate", ["release stock", "reject order"]))

## Step 5 · Review failed and successful checkouts

Both cases completed `create order`, `verify customer`, and `reserve stock` before authorization. On failure, reverse only the completed writes. On success, do not compensate anything. The model receives the case and skill, not the expected answer; the checker uses those labels only after generation. One repair is allowed after a broken response, then the run stops.

In [ ]:
cases = [("Payment failed", "compensate", ["release stock", "reject order"]),
         ("Payment succeeded", "complete", [])]
format_note = "Use exactly Context:, Decision:, Consequence:. Decision contains only compensate or complete."
for case, expected, actions in cases:
    prompt = f"{skills[choice]['body']}\n{format_note}\nCompleted: create order, verify customer, reserve stock. {case}."
    for attempt in range(2):
        adr = ask(prompt)
        problems = adr_problems(adr, expected, actions)
        if not problems:
            break
        prompt += f"\nYour last reply: {adr}\nFix these problems: {problems}. Return only three lines."
    print(f"{adr}\n-> {problems or 'passes'}\n")
    assert not problems, problems

## What you should see

The catalog contains three skills, but only `saga-adr` is loaded. The direct checker test rejects a plan in the wrong order. The failed checkout's accepted plan releases stock before rejecting the order; the successful checkout has no compensation. Neither plan runs a business action.

## Try it

1. Pass a plan in reversed order to the checker to `reject order; release stock`. Why is it rejected even though both action names occur?
2. Add `refund payment` after a failed authorization. Why is that not compensation for a completed local write?
3. Remove `reserve stock` from the completed steps. Which action must also disappear from the plan and checker expectation?
4. Change the task to reviewing a bounded context. Which catalog entry should load? Run Step 3 only and inspect the selected skill.
5. **Use it in your council.** Save this as a real `SKILL.md`, then add a saga step with an explicit compensation. Keep proposed plans separate from authorized tool execution.

## Takeaway

A skill packages a repeatable domain procedure. Progressive disclosure loads only the chosen procedure, and validation still checks its result. Saga compensation reverses completed writes; it is not an isolated distributed rollback.